## **Inputs for running the Regional Ocean Modeling System (ROMS): Indian Ocean Regional Grid, Initial, Boundary and Surface Forcing File Preparation in Google Colab**

## *Steps to Access Google Colab and the Notebook File*

Step 1: Open the link:
 https://github.com/prasannakanti/Data_Grid/blob/main/Grid_IC_BC_Preparation_Steps.ipynb

Step 2: Click the three dots in the top-right corner of the window → click Download.
The file Grid_IC_BC_Preparation_Steps.ipynb will be saved in your Downloads folder.

Step 3: Open Google Colab: https://colab.research.google.com/

Step 4: In Google Colab, go to
File → Upload notebook → Upload → Browse
Locate Grid_IC_BC_Preparation_Steps.ipynb in your Downloads folder and upload it.

### This is notebook 1 of 2

| | Notebook | What you do | Result |
|---|---|---|---|
| **1** | `Grid_IC_BC_Preparation_Steps.ipynb` (this one) | create and edit the grid, then initial, boundary and surface-forcing files | `roms_grd.nc`, `roms_ini.nc`, `roms_frc.nc`, `roms_bry.nc` in Google Drive → `MyDrive/ROMS_class/inputs` |
| **2** | `ROMS_class_2026.ipynb` | compile and run ROMS with **your** files (closed boundaries), then analyse | model output and plots |

Run every cell from top to bottom. The last cell saves your files to Google Drive; notebook 2 picks them up from there.

## **Step1** : Required python package roms-tools (ref: https://roms-tools.readthedocs.io/en/latest/)

The version is pinned, so everyone in the class gets the same behaviour.
If Colab asks you to **restart the session** after the install, click *Restart* and continue with the next cell.

In [ ]:
%pip install -q "roms-tools==5.2.2"

## Step 2: Download required files (Bathymetry file, Reanalysis Data)

In [ ]:
![ -d /content/Data_Grid ] || git clone https://github.com/prasannakanti/Data_Grid.git /content/Data_Grid

In [ ]:
# Change directory
%cd /content/Data_Grid

In [ ]:
!pwd

In [ ]:
!ls

## **Common settings**

Every file must use the **same `model_reference_date`**: ROMS reads all times as "seconds or days since this date".
If the initial file and the boundary file use different reference dates, the model starts at the wrong time relative to its forcing.

In [ ]:
import os, shutil
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime
from roms_tools import Grid, InitialConditions, BoundaryForcing, SurfaceForcing

DATA = '/content/Data_Grid'
OUT = f'{DATA}/output'
os.makedirs(OUT, exist_ok=True)

MODEL_REF = datetime(2000, 1, 1)          # one reference date for ALL files
INI_TIME = datetime(1993, 1, 16, 12)      # = the time stamp of the GLORYS January-1993 monthly mean
GLORYS_FILE = f'{DATA}/GLORY_Jan_1deg.nc'

## **Grid Preparation**

## **Visualization of bathymetry file**

In [ ]:
bathy = xr.open_dataset(f'{DATA}/SRTM15_V2.6_IO_1deg.nc').z

In [ ]:
bathy.plot()

## **Grid Preparation**

In [ ]:
grid = Grid(
    nx=91,              # ➤ Number of INTERIOR grid points in x (the file gets nx+2 = 93 rho-points)
    ny=61,              # ➤ Number of INTERIOR grid points in y (the file gets ny+2 = 63 rho-points)
    size_x=9100,        # ➤ Total horizontal domain size in the x-direction (in km) -> 100 km spacing
    size_y=6100,        # ➤ Total horizontal domain size in the y-direction (in km)
    center_lon=75,      # ➤ Longitude of the domain center (e.g., 75°E for Indian Ocean)
    center_lat=0,       # ➤ Latitude of the domain center (e.g., equatorial region)
    rot=0,              # ➤ Grid rotation angle (0° means east–west oriented grid)
    theta_s=5,          # ➤ Surface stretching parameter for vertical sigma levels
    theta_b=0.7,        # ➤ Bottom stretching parameter for vertical sigma levels
    hc=300,             # ➤ Critical depth controlling sigma coordinate stretching
    topography_source={
        "name": "SRTM15",                                     # ➤ Bathymetry dataset name
        "path": f"{DATA}/SRTM15_V2.6_IO_1deg.nc"              # ➤ File path to SRTM15 topography data
    },
    N=50,               # ➤ Number of vertical layers (higher = finer vertical resolution)
    close_narrow_channels=True,   # ➤ Close one-cell-wide channels a 1° grid cannot resolve
    verbose=True,       # ➤ Print progress and diagnostic information during grid generation
)

### Optional: edit the land–sea mask by hand

Click cells (or drag a box) to switch them between land and ocean, then press **Apply to grid**.
Newly opened ocean cells keep the smoothed bathymetry at that point, so check `h` there afterwards.

In [ ]:
%pip -q install ipywidgets

import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript
from google.colab import output
import uuid, json

output.enable_custom_widget_manager()

# Original and editable masks
mask_original = grid.ds["mask_rho"].values.copy().astype(np.int8)
mask_edit = mask_original.copy()
history = []

canvas_id = "roms_" + uuid.uuid4().hex[:8]

mode = widgets.ToggleButtons(
    options=[
        ("Toggle", "toggle"),
        ("Land (0)", "land"),
        ("Ocean (1)", "ocean")
    ],
    value="toggle",
    description="Mode:"
)

selection = widgets.ToggleButtons(
    options=[
        ("Single cell", "cell"),
        ("Box region", "box")
    ],
    value="cell",
    description="Select:"
)

undo_btn = widgets.Button(description="Undo")
reset_btn = widgets.Button(description="Reset")
apply_btn = widgets.Button(
    description="Apply to grid",
    button_style="success"
)
status = widgets.HTML(value="Ready.")

# Canvas: blue = ocean, tan = land
html = """
<div style="overflow:auto;max-width:100%">
<canvas id="CANVAS_ID"
style="border:1px solid #777;cursor:crosshair;touch-action:none">
</canvas>
</div>
<script>
(() => {
 const canvas = document.getElementById("CANVAS_ID");
 const ctx = canvas.getContext("2d");
 const nx = NX, ny = NY, cell = 10;
 canvas.width = nx * cell;
 canvas.height = ny * cell;

 let data = INITIAL_MASK;
 let mode = "toggle", selection = "cell";
 let start = null, end = null, dragging = false;

 function draw() {
   ctx.clearRect(0, 0, canvas.width, canvas.height);

   for (let i=0; i<ny; i++) {
     for (let j=0; j<nx; j++) {
       ctx.fillStyle = data[i][j] ? "#168bd2" : "#d8b36a";
       ctx.fillRect(j*cell, (ny-1-i)*cell, cell-0.5, cell-0.5);
     }
   }

   if (start && end && selection === "box") {
     ctx.strokeStyle = "red";
     ctx.lineWidth = 2;
     ctx.strokeRect(
       Math.min(start.x,end.x)*cell,
       (ny-1-Math.max(start.y,end.y))*cell,
       (Math.abs(start.x-end.x)+1)*cell,
       (Math.abs(start.y-end.y)+1)*cell
     );
   }
 }

 function getCell(e) {
   const r = canvas.getBoundingClientRect();
   return {
     x: Math.max(0, Math.min(nx-1,
       Math.floor((e.clientX-r.left)*canvas.width/r.width/cell))),
     y: Math.max(0, Math.min(ny-1,
       ny-1-Math.floor((e.clientY-r.top)*canvas.height/r.height/cell)))
   };
 }

 function sendEdit(a,b) {
   google.colab.kernel.invokeFunction(
     "roms.edit",
     [a.x,a.y,b.x,b.y,mode,selection],
     {}
   );
 }

 canvas.addEventListener("pointerdown", e => {
   e.preventDefault();
   canvas.setPointerCapture(e.pointerId);
   start = getCell(e);
   end = start;
   dragging = true;

   if (selection === "cell") {
     sendEdit(start,start);
     start = null;
     dragging = false;
   }
   draw();
 });

 canvas.addEventListener("pointermove", e => {
   if (dragging) {
     end = getCell(e);
     draw();
   }
 });

 canvas.addEventListener("pointerup", e => {
   if (!dragging) return;
   end = getCell(e);
   sendEdit(start,end);
   start = null;
   end = null;
   dragging = false;
   draw();
 });

 window["CANVAS_ID_update"] = (newData, newMode, newSelection) => {
   data = newData;
   mode = newMode;
   selection = newSelection;
   draw();
 };

 draw();
})();
</script>
"""

html = (html
    .replace("CANVAS_ID", canvas_id)
    .replace("NX", str(mask_edit.shape[1]))
    .replace("NY", str(mask_edit.shape[0]))
    .replace("INITIAL_MASK", json.dumps(mask_edit.tolist()))
)

canvas_out = widgets.Output()
with canvas_out:
    display(HTML(html))


def push_canvas():
    js = """
    window["CANVAS_ID_update"](
        NEW_DATA, NEW_MODE, NEW_SELECTION
    );
    """
    js = (js
        .replace("CANVAS_ID", canvas_id)
        .replace("NEW_DATA", json.dumps(mask_edit.tolist()))
        .replace("NEW_MODE", json.dumps(mode.value))
        .replace("NEW_SELECTION", json.dumps(selection.value))
    )
    display(Javascript(js))


def edit_callback(x1, y1, x2, y2, action, select):
    history.append(mask_edit.copy())

    i1, i2 = sorted([y1, y2])
    j1, j2 = sorted([x1, x2])

    if select == "cell":
        i2, j2 = i1, j1

    region = mask_edit[i1:i2+1, j1:j2+1]

    if action == "land":
        region[:] = 0
    elif action == "ocean":
        region[:] = 1
    else:
        region[:] = 1 - region

    status.value = (
        f"Edited rows {i1}–{i2}, columns {j1}–{j2}."
    )
    push_canvas()


output.register_callback("roms.edit", edit_callback)


def update_mode(change):
    push_canvas()

mode.observe(update_mode, names="value")
selection.observe(
    lambda change: push_canvas(), names="value"
)


def undo(_):
    if history:
        mask_edit[:] = history.pop()
        push_canvas()
        status.value = "Last edit undone."


def reset(_):
    history.append(mask_edit.copy())
    mask_edit[:] = mask_original
    push_canvas()
    status.value = "Original mask restored."


def apply(_):
    rho = mask_edit.astype(grid.ds["mask_rho"].dtype)
    grid.ds["mask_rho"].data = rho

    if "mask_u" in grid.ds:
        grid.ds["mask_u"].data = (
            rho[:, :-1] * rho[:, 1:]
        ).astype(grid.ds["mask_u"].dtype)

    if "mask_v" in grid.ds:
        grid.ds["mask_v"].data = (
            rho[:-1, :] * rho[1:, :]
        ).astype(grid.ds["mask_v"].dtype)

    if "mask_psi" in grid.ds:
        psi = (
            rho[:-1,:-1] * rho[1:,:-1] *
            rho[:-1,1:] * rho[1:,1:]
        )
        grid.ds["mask_psi"].data = psi.astype(
            grid.ds["mask_psi"].dtype
        )

    status.value = "Mask applied to grid. Review bathymetry before saving."


undo_btn.on_click(undo)
reset_btn.on_click(reset)
apply_btn.on_click(apply)

display(widgets.VBox([
    widgets.HBox([mode, selection]),
    widgets.HBox([undo_btn, reset_btn, apply_btn]),
    status,
    canvas_out
]))


## **Visualization and Diagnostics**
In this section, we visualize the prepared grid and datasets to ensure correctness.  
Common diagnostics include:
- Plotting bathymetry and land-sea masks.  
- Checking smoothness of topography.  
- Verifying boundaries and vertical levels.

In [ ]:
grid.ds.mask_rho.plot()

In [ ]:
grid.ds.attrs

In [ ]:
grid.ds.mask_u.plot()

In [ ]:
grid.ds.h.plot(robust=True)

In [ ]:
plt.plot(grid.ds.sigma_r, grid.ds.Cs_r)
plt.xlabel("sigma_r")
plt.ylabel("Cs_r")
plt.title("Vertical stretching function at rho-points")

In [ ]:
grid.plot_vertical_coordinate(eta=31)                          # a west–east section (eta_rho goes 0..62)

In [ ]:
grid.plot_vertical_coordinate(xi=50, max_nr_layer_contours=10) # a south–north section (xi_rho goes 0..92)

In [ ]:
# Topography steepness: keep rx0 below about 0.2
h, m = grid.ds.h.values, grid.ds.mask_rho.values
rx0 = max(np.nanmax(np.abs(np.diff(h, axis=1)) / (h[:, 1:] + h[:, :-1]) * m[:, 1:] * m[:, :-1]),
          np.nanmax(np.abs(np.diff(h, axis=0)) / (h[1:] + h[:-1]) * m[1:] * m[:-1]))
print(f"grid {grid.ds.sizes['xi_rho']} x {grid.ds.sizes['eta_rho']} rho-points | "
      f"ocean points {int(m.sum())} | h {h.min():.0f}–{h.max():.0f} m | max rx0 = {rx0:.2f}")

## **Save the Grid**

In [ ]:
grid.save(f'{OUT}/IO_grd.nc')

## **Initial Condition Preparation**
This section prepares the initial condition (`ini`) file required by ROMS.
It involves:
1. Loading GLORYS reanalysis data containing temperature, salinity, velocity and sea level.
2. Filling gaps below the GLORYS sea floor (next cell).
3. Interpolating these fields to the ROMS grid and its s-levels.
4. Saving the result as `IO_ini.nc`, which initializes the ocean state at the model start time.

### ⚠️ Fill the sea-floor gaps in GLORYS first

The smoothed 1° model bathymetry is much deeper than GLORYS over small banks (Seychelles Bank, Saya de Malha).
Below the GLORYS sea floor, roms-tools copies the **deepest available value downward**, which puts ~25 °C surface water at 1,000 m right next to ~10 °C water.
ROMS blows up within a few days from that density contrast.
**Fix:** at every depth level, fill missing ocean points with the nearest GLORYS value **at the same depth**.

In [ ]:
from scipy import ndimage

def fill_seafloor_gaps(ds, variables=('thetao', 'so', 'uo', 'vo')):
    # Columns that are ocean at the surface but end above the model sea floor: fill each missing
    # level with the nearest ocean value at the SAME depth (instead of the value from above).
    ds = ds.load().copy()
    for v in variables:
        a = ds[v].values                                  # (time, depth, lat, lon)
        ocean = np.isfinite(a[:, 0])                      # ocean at the surface
        for t in range(a.shape[0]):
            for k in range(1, a.shape[1]):
                lev = a[t, k]; valid = np.isfinite(lev)
                if not valid.any(): break                 # below the deepest ocean point
                gap = ocean[t] & ~valid
                if gap.any():
                    _, (jj, ii) = ndimage.distance_transform_edt(~valid, return_indices=True)
                    lev[gap] = lev[jj[gap], ii[gap]]
        ds[v].values = a
    return ds

path = f'{DATA}/GLORYS_filled.nc'
fill_seafloor_gaps(xr.open_dataset(GLORYS_FILE)).to_netcdf(path)
print('written', path)

In [ ]:
initial_conditions = InitialConditions(
    grid=grid,
    ini_time=INI_TIME,                      # must match a time in the GLORYS file (16 Jan 1993 12:00)
    source={"name": "GLORYS", "path": path},
    model_reference_date=MODEL_REF,         # the SAME reference date as the boundary and surface forcing
    use_dask=True,
)

In [ ]:
initial_conditions.ds.ubar.compute()

In [ ]:
# Check: largest temperature jump between neighbouring ocean cells below 500 m (should be a few °C)
T = initial_conditions.ds.temp.isel(ocean_time=0).values
g = grid.ds
z = ((g.hc * g.sigma_r + g.h * g.Cs_r) / (g.hc + g.h) * g.h).values
mk = g.mask_rho.values
deep = (z[:, :, 1:] < -500) & (z[:, :, :-1] < -500) & (mk[:, 1:] * mk[:, :-1] == 1)
dT = np.nanmax(np.where(deep, np.abs(np.diff(T, axis=2)), 0))
print(f'max deep lateral temperature jump: {dT:.1f} °C', '✅' if dT < 6 else '❌ check the GLORYS gap filling')

In [ ]:
initial_conditions.plot("temp", s=-1)

In [ ]:
initial_conditions.save(f'{OUT}/IO_ini.nc')

## **Boundary Condition Preparation**
This step defines the boundary conditions for the ROMS domain.  
The procedure includes:
1. Extracting lateral boundaries from global datasets.
2. Performing horizontal and vertical interpolation.
3. Writing the final boundary condition file `IO_bry.nc` for ROMS input.

**Time range:** ROMS interpolates boundary values in time, so the boundary file must cover the whole run.
The demo GLORYS file has **one** record (January 1993), so the boundary file below holds a single, constant state.
For a real run, download one GLORYS monthly mean per month of the run (plus one month before and after) and set `start_time` / `end_time` to span them.

In [ ]:
start_time = INI_TIME
end_time = INI_TIME

In [ ]:
boundary_forcing = BoundaryForcing(
    grid=grid,
    start_time=start_time,
    end_time=end_time,
    boundaries={
        "south": True,
        "east": True,
        "north": False,  # northern boundary excluded
        "west": False,   # western boundary excluded
    },
    source={"name": "GLORYS", "path": path},
    apply_2d_horizontal_fill=True,  # the eastern boundary is cut by Indonesian islands
    model_reference_date=MODEL_REF, # the SAME reference date as the initial conditions
    use_dask=True,
)

In [ ]:
boundary_forcing.physics.plot("temp_south", time=0, layer_contours=True)

In [ ]:
boundary_forcing.physics.plot("temp_east", time=0, layer_contours=True)

In [ ]:
boundary_forcing.save(f'{OUT}/IO_bry.nc', group=False)   # group=False: one file, not one per month

## **Forcing File Preparation From ERA5** (Ref: https://roms-tools.readthedocs.io/en/latest/surface_forcing.html)

roms-tools needs these ERA5 variables in one file: **`u10, v10, ssr, strd, t2m, d2m, tp, sst`**
(`ERA5_T2M.nc` in the repository holds only `t2m`, which is why the original cell failed).

**Monthly means need one extra step.** roms-tools assumes **hourly** ERA5, where radiation and rain are accumulated over **one hour**.
In the *monthly averaged reanalysis* product they are accumulated over **one day**, so we divide `ssr`, `strd` and `tp` by 24.
We also rename `valid_time` → `time`, and stamp each monthly mean on the 15th of the month instead of the 1st.

Choose the source:
* **Option A**: your own download from the Climate Data Store (see the exercise below), saved as `ERA5_monthly_raw.nc`.
* **Option B**: a practice file in the same format, rebuilt from `ERA5.nc` in this repository, so the notebook runs without a CDS account.

In [ ]:
ERA5_OPTION = 'B'                                   # 'A' = your CDS download, 'B' = practice file
ERA5_RAW = f'{DATA}/ERA5_monthly_raw.nc'

In [ ]:
# ---- Option B: practice file in CDS format, rebuilt from ERA5.nc (12 monthly means of 1993) ----
if ERA5_OPTION == 'B':
    e = xr.open_dataset(f'{DATA}/ERA5.nc').drop_vars('time_bnds')
    e = xr.concat([e.isel(time=[-1]).assign_coords(time=[pd.Timestamp('1992-12-01')]), e,     # Dec-1992 := Dec-1993
                   e.isel(time=[0]).assign_coords(time=[pd.Timestamp('1994-01-01')])], 'time')  # Jan-1994 := Jan-1993
    vap = e.Q2M * (e.msl / 100) / (0.622 + 0.378 * e.Q2M)              # vapour pressure (hPa)
    x = np.log(vap / 6.1121); d2m = 240.97 * x / (17.502 - x)          # dew point (°C)
    sea = xr.open_dataset(GLORYS_FILE).thetao.isel(time=0, depth=0).interp(
          longitude=e.lon.values, latitude=e.lat.values, method='nearest').notnull().values
    raw = xr.Dataset({'u10': e.u10, 'v10': e.v10, 'msl': e.msl, 't2m': e.t2m + 273.15, 'd2m': d2m + 273.15,
                      'ssr': 0.94 * e.ssrd * 86400, 'strd': e.strd * 86400, 'tp': e.tp / 1000 * 86400,
                      'sst': (e.t2m + 273.15).where(xr.DataArray(sea, dims=('lat', 'lon')))})
    raw.rename({'time': 'valid_time', 'lat': 'latitude', 'lon': 'longitude'}).astype('float32').to_netcdf(ERA5_RAW)
    print('practice file written:', ERA5_RAW)

In [ ]:
# ---- Prepare the monthly file for roms-tools (both options) ----
era = xr.open_dataset(ERA5_RAW)
era = era.rename({'valid_time': 'time'}).drop_vars(['number', 'expver'], errors='ignore').sortby('latitude')
for v in ['ssr', 'strd', 'tp']:
    era[v] = era[v] / 24.0                                             # per-day -> per-hour accumulation
era = era.assign_coords(time=pd.to_datetime(era.time.values) + pd.Timedelta(days=14))   # 1st -> 15th
path = f'{DATA}/ERA5_for_romstools.nc'
era.to_netcdf(path)
print(era.time.values.astype('datetime64[D]'))

In [ ]:
# The forcing period must cover the run, which starts on INI_TIME (16 Jan 1993)
start_time = datetime(1992, 12, 16)
end_time = datetime(1994, 1, 14)

In [ ]:
surface_forcing_kwargs = {
    "grid": grid,
    "start_time": start_time,
    "end_time": end_time,
    "model_reference_date": MODEL_REF,   # the SAME reference date as the other files
}

In [ ]:
%%time
surface_forcing_from_local_data = SurfaceForcing(
    **surface_forcing_kwargs,
    source={"name": "ERA5", "path": path},
    use_dask=True,
)

In [ ]:
# Sanity check: domain-mean values (expected over a tropical ocean: net SW 180–260 W/m², rain 3–6 mm/day)
sf = surface_forcing_from_local_data.ds
ocean = grid.ds.mask_rho == 1
chk = pd.DataFrame({'net SW (W/m2)': sf.swrad.where(ocean).mean(('eta_rho', 'xi_rho')),
                    'down LW (W/m2)': sf.lwrad.where(ocean).mean(('eta_rho', 'xi_rho')),
                    'Tair (C)': sf.Tair.where(ocean).mean(('eta_rho', 'xi_rho')),
                    'rain (mm/day)': 10 * sf.rain.where(ocean).mean(('eta_rho', 'xi_rho'))},
                   index=pd.to_datetime(sf.abs_time.values).strftime('%b-%Y')).round(1)
print(chk)

In [ ]:
surface_forcing_from_local_data.plot("uwnd", time=1)

In [ ]:
surface_forcing_from_local_data.save(f'{OUT}/IO_frc.nc', group=False)   # one file, not one per year

# **Exercise**
1. Download the required variables for 1 year (plus December of the year before and January of the year after) from
   https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=download
   * Product type: *Monthly averaged reanalysis*
   * Variables: 10m u- and v-component of wind, 2m temperature, 2m dewpoint temperature, surface net solar radiation,
     surface thermal radiation downwards, total precipitation, sea surface temperature
   * Sub-region: North 30°, West 30°, South −30°, East 120°; format NetCDF
2. Upload the file here, rename it `ERA5_monthly_raw.nc`, set `ERA5_OPTION = 'A'`, and re-run the surface-forcing cells.
3. Change `start_time`, `end_time` (and `INI_TIME` with a matching GLORYS file) to your year.

## **Prepare the files for ROMS and save them to Google Drive**

The files above are in **roms-tools (UCLA-ROMS)** format. Our class model is **Rutgers ROMS** (`ROMS_class_2026.ipynb`), which expects:

| | roms-tools gives | Rutgers ROMS needs |
|---|---|---|
| Grid | ρ, u, v points | also ψ-points, `dndx`, `dmde`, `xl`, `el`, Rutgers dimension names |
| Initial | time relative to `MODEL_REF` | time in seconds since 1970-01-01, plus the s-coordinate parameters |
| Forcing | `uwnd, vwnd, Tair, qair, swrad, lwrad, rain` (cm/day), no air pressure | `u10, v10, t2m, Q2M, ssrd, strd, tp` (kg m⁻² s⁻¹) **and** `msl` |

The next cell converts the grid, initial and forcing files and writes them as NetCDF-3, which ROMS (Fortran) always reads correctly.
The class run keeps **all four boundaries closed** (walls), so ROMS does not read the boundary file; we still copy it to Drive.

Each Colab notebook runs on its own machine, so the files go to **Google Drive** (`MyDrive/ROMS_class/inputs`), where `ROMS_class_2026.ipynb` finds them.

In [ ]:
import shutil, roms_tools

def write_nc3(ds, fname):
    # NetCDF-3 written with scipy: text attributes stay plain characters, which ROMS can read
    ds = ds.copy()
    for v in ds.variables:
        ds[v].encoding = {}
        if ds[v].dtype.kind in 'iu' and ds[v].dtype.itemsize == 8:
            ds[v] = ds[v].astype('int32')
    ds.to_netcdf(fname, format='NETCDF3_64BIT', engine='scipy')
    print(f'written {fname}  ({os.path.getsize(fname) / 1e6:.1f} MB)')

def _u(a):   return 0.5 * (a[:, :-1] + a[:, 1:])
def _v(a):   return 0.5 * (a[:-1, :] + a[1:, :])
def _psi(a): return 0.25 * (a[:-1, :-1] + a[1:, :-1] + a[:-1, 1:] + a[1:, 1:])
T1970 = pd.Timestamp('1970-01-01')

def grid_for_rutgers(grid):
    g = grid.ds
    m, h = g.mask_rho.values.astype('f8'), g.h.values.astype('f8')
    lon, lat, pm, pn = g.lon_rho.values, g.lat_rho.values, g.pm.values, g.pn.values
    R, U, V, P = ('eta_rho', 'xi_rho'), ('eta_u', 'xi_u'), ('eta_v', 'xi_v'), ('eta_psi', 'xi_psi')
    ds = xr.Dataset({
        'xl': ((), float(np.mean(np.sum(1 / pm[:, 1:-1], axis=1)))),
        'el': ((), float(np.mean(np.sum(1 / pn[1:-1, :], axis=0)))),
        'spherical': ((), np.array(b'T')),
        'h': (R, h), 'f': (R, g.f.values), 'pm': (R, pm), 'pn': (R, pn),
        'dndx': (R, np.gradient(1 / pn, axis=1)), 'dmde': (R, np.gradient(1 / pm, axis=0)),
        'angle': (R, g.angle.values),
        'lon_rho': (R, lon), 'lat_rho': (R, lat), 'lon_u': (U, _u(lon)), 'lat_u': (U, _u(lat)),
        'lon_v': (V, _v(lon)), 'lat_v': (V, _v(lat)), 'lon_psi': (P, _psi(lon)), 'lat_psi': (P, _psi(lat)),
        'mask_rho': (R, m), 'mask_u': (U, m[:, :-1] * m[:, 1:]), 'mask_v': (V, m[:-1, :] * m[1:, :]),
        'mask_psi': (P, m[:-1, :-1] * m[1:, :-1] * m[:-1, 1:] * m[1:, 1:]),
    }, attrs={'type': 'ROMS grid file', 'title': 'Indian Ocean grid, roms-tools ' + roms_tools.__version__})
    ds['h'].attrs = {'long_name': 'bathymetry at RHO-points', 'units': 'meter'}
    return ds

def ini_for_rutgers(ic, grid):
    d, g = ic.ds, grid.ds
    ds = xr.Dataset({'Vtransform': ((), 2), 'Vstretching': ((), 4),
                     'theta_s': ((), float(grid.theta_s)), 'theta_b': ((), float(grid.theta_b)),
                     'Tcline': ((), float(grid.hc)), 'hc': ((), float(grid.hc)),
                     's_rho': ('s_rho', g.sigma_r.values.astype('f8')), 'Cs_r': ('s_rho', g.Cs_r.values.astype('f8')),
                     's_w': ('s_w', g.sigma_w.values.astype('f8')), 'Cs_w': ('s_w', g.Cs_w.values.astype('f8'))})
    for v in ['zeta', 'temp', 'salt']:
        ds[v] = d[v].drop_vars(['abs_time', 'ocean_time'], errors='ignore')
    for v in ['u', 'ubar']:
        ds[v] = d[v].drop_vars(['abs_time', 'ocean_time'], errors='ignore').rename(eta_rho='eta_u')
    for v in ['v', 'vbar']:
        ds[v] = d[v].drop_vars(['abs_time', 'ocean_time'], errors='ignore').rename(xi_rho='xi_v')
    secs = (pd.Timestamp(INI_TIME) - T1970).total_seconds()      # absolute time, independent of MODEL_REF
    ds = ds.assign_coords(ocean_time=('ocean_time', [secs]))
    ds['ocean_time'].attrs = {'long_name': 'initial time', 'units': 'seconds since 1970-01-01 00:00:00', 'calendar': 'gregorian'}
    ds.attrs = {'type': 'ROMS initial file', 'source': 'GLORYS via roms-tools ' + roms_tools.__version__}
    return ds

def frc_for_rutgers(sf, era, grid):
    # names and units match the varinfo.yaml settings in ROMS_class_2026.ipynb
    d, g = sf.ds, grid.ds
    R = ('time', 'eta_rho', 'xi_rho')
    when = pd.to_datetime(d.abs_time.values)
    msl = (era.msl.sel(time=when).interp(latitude=g.lat_rho, longitude=g.lon_rho)   # roms-tools has no air pressure
              .transpose('time', 'eta_rho', 'xi_rho').values)
    ds = xr.Dataset({
        'u10': (R, d.uwnd.values), 'v10': (R, d.vwnd.values),
        't2m': (R, d.Tair.values), 'Q2M': (R, d.qair.values),
        'ssrd': (R, d.swrad.values / 0.94),   # net shortwave; varinfo.yaml multiplies 'ssrd' by 0.94 again
        'strd': (R, d.lwrad.values),          # downward longwave (ROMS: LONGWAVE_OUT)
        'tp': (R, d.rain.values * 10.0 / 86400.0),   # cm/day -> kg m-2 s-1
        'msl': (R, msl),
    })
    ds = ds.assign_coords(time=('time', ((when - T1970) / pd.Timedelta(days=1)).values.astype('f8')))
    ds['time'].attrs = {'long_name': 'forcing time', 'units': 'days since 1970-01-01 00:00:00', 'calendar': 'gregorian'}
    ds.attrs = {'type': 'ROMS forcing file', 'source': 'ERA5 monthly means via roms-tools ' + roms_tools.__version__}
    return ds

ROMS_DIR = f'{OUT}/for_roms'
os.makedirs(ROMS_DIR, exist_ok=True)
write_nc3(grid_for_rutgers(grid), f'{ROMS_DIR}/roms_grd.nc')
write_nc3(ini_for_rutgers(initial_conditions, grid), f'{ROMS_DIR}/roms_ini.nc')
write_nc3(frc_for_rutgers(surface_forcing_from_local_data, era, grid), f'{ROMS_DIR}/roms_frc.nc')
shutil.copy(f'{OUT}/IO_bry.nc', f'{ROMS_DIR}/roms_bry.nc')      # not used: the class run has closed boundaries

In [ ]:
# Quick check of what ROMS will read
ini_chk = xr.open_dataset(f'{ROMS_DIR}/roms_ini.nc')
frc_chk = xr.open_dataset(f'{ROMS_DIR}/roms_frc.nc')
print('grid  :', grid.ds.sizes['xi_rho'], 'x', grid.ds.sizes['eta_rho'], 'rho-points,', grid.N, 'levels')
print('start :', ini_chk.ocean_time.values[0])
print('forcing records:', frc_chk.time.values[0], '...', frc_chk.time.values[-1], f'({frc_chk.sizes["time"]})')
print('s-coordinates: Vtransform 2, Vstretching 4, theta_s', grid.theta_s, ', theta_b', grid.theta_b, ', hc', grid.hc)

### Save to Google Drive

Run this cell, allow access to your Drive, then open **`ROMS_class_2026.ipynb`** (notebook 2) in Colab.
It copies these files from `MyDrive/ROMS_class/inputs` automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ROMS_class/inputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
for f in ['roms_grd.nc', 'roms_ini.nc', 'roms_frc.nc', 'roms_bry.nc']:
    shutil.copy(f'{ROMS_DIR}/{f}', DRIVE_DIR)
print('Saved to', DRIVE_DIR, ':', sorted(os.listdir(DRIVE_DIR)))